# Day 1 — How LLMs Actually Work + First API Call
---
> **Phase:** 1 — LLM Foundations  
> **Status:** ✅ Complete

## 🧠 Key Concepts

### 1. Why Transformers Replaced RNNs & LSTMs

**Problem 1 — Vanishing Gradients (RNN)**
- RNNs process one token at a time, passing a hidden state forward
- During backpropagation, gradients shrink exponentially over time steps
- By the time gradient reaches early tokens → effectively zero
- Result: Model learns nothing about long-range dependencies

**Problem 2 — Sequential Computation (LSTM)**
- LSTMs fixed memory with gates (forget, input, output)
- BUT: Still processes token 2 only after token 1 is done
- Cannot parallelize → devastatingly slow on long sequences
- Training large models on LSTMs was practically infeasible

**Transformer Solution — Self-Attention**
> What if every token could directly look at every other token simultaneously?

```
RNN/LSTM   → Sequential, hidden state chain, forgets early tokens
Transformer → Parallel, every token sees every token, nothing forgotten
Self-Attention → The mechanism that makes this possible
```

### 2. What Happens When You Type a Prompt — End to End

**Stage 1 — Tokenization**
- Text enters the model as tokens, not words
- A token = a chunk of text (word, subword, punctuation, space)
- Example:
  - `"I love machine learning"` → `["I", " love", " machine", " learning"]` = 4 tokens
  - `"Unbelievably"` → `["Un", "believ", "ably"]` = 3 tokens
- Context windows are measured in tokens, not words
- GPT-4o = 128,000 token context window ≈ 96,000 words

**Stage 2 — Embedding the Tokens**
- Each token → high-dimensional vector
- Unlike Word2Vec (static), Transformer embeddings are contextual:
  - `"bank"` near `"river"` ≠ `"bank"` near `"money"`

**Stage 3 — Self-Attention**
- For every token: *"Which other tokens should I pay attention to, and how much?"*
- Computes attention scores between every pair of tokens simultaneously
- Each token's final representation = weighted blend of all other tokens
- Example: In *"The animal didn't cross the street because it was too tired"*
  - When processing `"it"` → attends strongly to `"animal"`

**Stage 4 — GPT = Decoder-Only Transformer**
- Trained on one task: *Given all previous tokens, predict the next token*
- During inference: predicts one token at a time, feeds it back as input
- This is called **autoregressive generation**

**Stage 5 — Temperature & Top-P**

| Parameter | Value | Effect |
|-----------|-------|--------|
| Temperature | 0 | Always pick highest probability token (deterministic) |
| Temperature | 1 | Sample proportionally (balanced) |
| Temperature | 1.5+ | Flatten distribution (creative, random) |
| Top-P | 0.9 | Only sample from top 90% probability mass |

```
Use temperature=0  → classification, structured output, factual tasks
Use temperature>0  → creative writing, brainstorming, open-ended tasks
```

### 3. System Messages
- A system message is NOT architecturally special
- It is just text placed at the BEGINNING of the context window
- The model sees it all as one continuous token sequence:
```
[SYSTEM TOKENS] You are an expert Python teacher...
[USER TOKENS]   What is a token in LLMs?
[ASSISTANT TOKENS] ← model predicts from here
```
- ⚠️ System message consumes tokens from your context window

## 💻 Code

In [ ]:
# Day 1 — First Raw API Call
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what a token is in LLMs in 3 sentences."}
    ],
    temperature=0.7,   # controls randomness
    max_tokens=500     # token budget for response
)

print(response.choices[0].message.content)

## 🔬 Experiments & Observations

| Experiment | Observation |
|-----------|-------------|
| temperature=0, run 3x | Same output every time — picks highest probability token |
| temperature=1.5, run 3x | Different output every time — distribution flattened |
| max_tokens=20 | Response gets cut off — hits token budget |
| Poem at temp=0 vs 1.5 | temp=0 same every time, temp=1.5 different every time |

## ✅ Day 1 Summary

```
✓ Why Transformers replaced RNNs (derived independently)
✓ What tokens are and why context windows are measured in them
✓ How temperature and top-p control generation
✓ How to make a raw API call without any framework
✓ How to read and debug API errors (400, 401, 429, 500)
✓ Proper project structure and secret management (.env + .gitignore)
✓ System messages are just tokens at the start of context window
```

## ⚠️ Common Mistakes to Avoid

1. **Never hardcode API keys** — always use `.env` + `python-dotenv`
2. **Never commit `.env` to git** — always add to `.gitignore`
3. **UV creates its own `.gitignore` inside `.venv`** — that only protects the venv folder, NOT your `.env`
4. **`.gitignore` must be at project root**, not inside `.venv`